this is the problem of: 

https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview

this is a competition of prediction of the houses. there are

Adem Cayir - A00046645 - TU060 - Data Science Master Program (PartTime)


In [102]:
import os
import requests

# here, downloading the data files. I uploaded to github, and use public
# address. so no need any accoujnt
def download_if_missing(url: str, local_file: str):
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    if os.path.exists(local_file):
        print(f"{local_file} already exists. Skipping download.")
    else:
        print(f"{local_file} not found. Downloading from {url}...")
        response = requests.get(url)
        response.raise_for_status()
        with open(local_file, "wb") as f:
            f.write(response.content)
        print("Download complete.")

download_if_missing("https://raw.githubusercontent.com/ademcayir/HousePricesPrediction/refs/heads/main/data/train.csv","data/train.csv")
download_if_missing("https://raw.githubusercontent.com/ademcayir/HousePricesPrediction/refs/heads/main/data/data_description.txt","data/data_description.txt")
download_if_missing("https://raw.githubusercontent.com/ademcayir/HousePricesPrediction/refs/heads/main/data/test.csv","data/test.csv")
                    


data/train.csv already exists. Skipping download.
data/data_description.txt already exists. Skipping download.
data/test.csv already exists. Skipping download.


In [103]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col, sum, expr
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PCA
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.sql.functions import mean
from pyspark.sql import DataFrame
import pandas as pd # this is only for exporting csv file

#Create a Spark Session
spark = SparkSession.builder \
        .appName('House Price Prediction') \
        .master('local[*]') \
        .getOrCreate()

In [104]:


# in this dataset, there are too many column. 
# I would like to ignore categorical column, since when I investigate them, 
# I see mostly they are insignificant
# 
# side note: I first include only numerical features. but, outcome was not perfect. 
# Kaggle score was 0.181. as my investigation, it is not bad, but not good as well
# so that, later on, I added categorical features as well
# id in here is not a feature sent to the model
# SalePrice is the target feature
# all string features are categorical
# detail of the categories are documented in data/data_description.txt file
features_cols = [
    {"Id":"int"},
    {"LotArea":"int"},
    {"YearBuilt":"int"},
    {"1stFlrSF":"int"},
    {"2ndFlrSF":"int"},
    {"FullBath":"int"},
    {"BedroomAbvGr":"int"},
    {"TotRmsAbvGrd":"int"},
    {"OverallQual":"int"},
    {"OverallCond":"int"},
    {"GarageCars":"int"},
    {"GarageArea":"int"},
    {"TotalBsmtSF":"int"},
    {"GrLivArea":"int"},
    {"YearRemodAdd":"int"},
    {"SalePrice":"int"},
    {"MSSubClass": "string" },
    {"MSZoning":"string"},
    {"Street":"string"},
    {"LotShape": "string"},
    {"BldgType": "string"},
    {"HouseStyle": "stirng"},
    {"RoofStyle":"string"},
    {"Neighborhood": "string"},
    {"LotConfig": "string"},
    {"SaleCondition": "string"},
    {"SaleType": "string"},
    {"MiscFeature": "string"},
    {"Fence": "string"},
    {"PoolQC": "string"},
    {"PavedDrive": "string"},
    {"GarageCond": "string"},
    {"GarageQual": "string"},
    {"GarageFinish": "string"},
    {"KitchenQual": "string"},
    {"Electrical": "string"},
    {"CentralAir": "string"},
    {"HeatingQC": "string"},
    {"Heating": "string"},
    {"GarageType":"string"}
]

# here I load the data by types
# I forcefully used the type, becuase 
# I see when I try to load test.csv, 
# it loads integer columns as string
# so, I added this function to force column type
# I also realized, text.csv file has missing values
# so, missing values causing error when model is running
# so that, I try to cast to int first. but if it is null, I leave it as it is and later on,
# I set mean of the feature for empty values
def load_data(file: str, includeSalePrice: bool = True):
    raw_df = spark.read.csv(file, header=True)
    # here, I only read the columns which I define in feature_cols
    filter = []
    for item in features_cols:
        for col_name, col_type in item.items():
            # I added this, becuase 
            if includeSalePrice == True or col_name != "SalePrice":
                if col_type == "int":
                    # numeric: safe cast, leave NULLs
                    filter.append(expr(f"try_cast(`{col_name}` as int)").alias(col_name))
                else:
                    # categorical: keep as string
                    filter.append(col(col_name))
    filter_df = raw_df.select(filter)
    return filter_df

df = load_data("data/train.csv")
df.show(10)
# I split first dataset for test and train data. 
# be aware, there is test.csv file, but this is for kaggle submission. 
# it does not have the values. 
# so, I need to use train.csv file to decide which model is better
# so that, I need to read train.csv ve use it for both training and testing
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)


+---+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+---------+----------+--------+------+--------+--------+----------+---------+------------+---------+-------------+--------+-----------+-----+------+----------+----------+----------+------------+-----------+----------+----------+---------+-------+----------+
| Id|LotArea|YearBuilt|1stFlrSF|2ndFlrSF|FullBath|BedroomAbvGr|TotRmsAbvGrd|OverallQual|OverallCond|GarageCars|GarageArea|TotalBsmtSF|GrLivArea|YearRemodAdd|SalePrice|MSSubClass|MSZoning|Street|LotShape|BldgType|HouseStyle|RoofStyle|Neighborhood|LotConfig|SaleCondition|SaleType|MiscFeature|Fence|PoolQC|PavedDrive|GarageCond|GarageQual|GarageFinish|KitchenQual|Electrical|CentralAir|HeatingQC|Heating|GarageType|
+---+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+---------+-----

In [105]:
# checking the schema, even though we set all the column types
df.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- LotArea: integer (nullable = true)
 |-- YearBuilt: integer (nullable = true)
 |-- 1stFlrSF: integer (nullable = true)
 |-- 2ndFlrSF: integer (nullable = true)
 |-- FullBath: integer (nullable = true)
 |-- BedroomAbvGr: integer (nullable = true)
 |-- TotRmsAbvGrd: integer (nullable = true)
 |-- OverallQual: integer (nullable = true)
 |-- OverallCond: integer (nullable = true)
 |-- GarageCars: integer (nullable = true)
 |-- GarageArea: integer (nullable = true)
 |-- TotalBsmtSF: integer (nullable = true)
 |-- GrLivArea: integer (nullable = true)
 |-- YearRemodAdd: integer (nullable = true)
 |-- SalePrice: integer (nullable = true)
 |-- MSSubClass: string (nullable = true)
 |-- MSZoning: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- LotShape: string (nullable = true)
 |-- BldgType: string (nullable = true)
 |-- HouseStyle: string (nullable = true)
 |-- RoofStyle: string (nullable = true)
 |-- Neighborhood: string (nullable

In [106]:
# checking null or duplicate or empty values
df.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
]).show()
# so that, there is not row which has any null value

+---+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+---------+----------+--------+------+--------+--------+----------+---------+------------+---------+-------------+--------+-----------+-----+------+----------+----------+----------+------------+-----------+----------+----------+---------+-------+----------+
| Id|LotArea|YearBuilt|1stFlrSF|2ndFlrSF|FullBath|BedroomAbvGr|TotRmsAbvGrd|OverallQual|OverallCond|GarageCars|GarageArea|TotalBsmtSF|GrLivArea|YearRemodAdd|SalePrice|MSSubClass|MSZoning|Street|LotShape|BldgType|HouseStyle|RoofStyle|Neighborhood|LotConfig|SaleCondition|SaleType|MiscFeature|Fence|PoolQC|PavedDrive|GarageCond|GarageQual|GarageFinish|KitchenQual|Electrical|CentralAir|HeatingQC|Heating|GarageType|
+---+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+---------+-----

In [107]:
print(f"before dropping duplicates, number of rows: {df.count()}")
df = df.dropDuplicates()
print(f"after dropping duplicates, number of rows: {df.count()}")
# so that there is 1460 rows and there is no duplicate row


before dropping duplicates, number of rows: 1460
after dropping duplicates, number of rows: 1460


In [108]:
df.describe().show()

+-------+----------------+------------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------+------+--------+--------+----------+---------+------------+---------+-------------+--------+-----------+-----+------+----------+----------+----------+------------+-----------+----------+----------+---------+-------+----------+
|summary|              Id|           LotArea|         YearBuilt|         1stFlrSF|          2ndFlrSF|          FullBath|      BedroomAbvGr|      TotRmsAbvGrd|       OverallQual|       OverallCond|        GarageCars|        GarageArea|       TotalBsmtSF|        GrLivArea|      YearRemodAdd|         SalePrice|        MSSubClass|MSZoning|Street|LotShape|BldgType|HouseStyle|RoofStyle|Neighborhood|LotConfig|SaleCondition|SaleType|MiscFea

In [109]:
# just printing columns 
print(df.columns)

['Id', 'LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'OverallQual', 'OverallCond', 'GarageCars', 'GarageArea', 'TotalBsmtSF', 'GrLivArea', 'YearRemodAdd', 'SalePrice', 'MSSubClass', 'MSZoning', 'Street', 'LotShape', 'BldgType', 'HouseStyle', 'RoofStyle', 'Neighborhood', 'LotConfig', 'SaleCondition', 'SaleType', 'MiscFeature', 'Fence', 'PoolQC', 'PavedDrive', 'GarageCond', 'GarageQual', 'GarageFinish', 'KitchenQual', 'Electrical', 'CentralAir', 'HeatingQC', 'Heating', 'GarageType']


In [110]:

# first, I split the feature set by integer
# and categorical
# I filter integer types as numberic_cols. just note that, I dont include 
# Id , which is identifier of the instance, and SalePrice is the target feature
numeric_cols = [col_name for item in features_cols for col_name, col_type in item.items() if col_type == "int" and col_name not in ["Id","SalePrice"]]
# in this dataset, all string types are categorical
categorical_cols = [col_name for item in features_cols for col_name, col_type in item.items() if col_type == "string"]

# first, creating indexer for categorical_cols
indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_index", handleInvalid="keep")
    for c in categorical_cols
]

# then use OneHotEncoder to conver categories to numeric values
encoders = [
    OneHotEncoder(inputCols=[c+"_index"], outputCols=[c+"_vec"])
    for c in categorical_cols
]

# creating assmebler with numeric_cols as it is, and including categorical features as encoder and indexer
# and group it as 'features'
assembler = VectorAssembler(
    inputCols=numeric_cols + [c+"_vec" for c in categorical_cols],
    outputCol="features"
)


In [111]:

# first, I want to check the model with lineer regression, before checking other models 
lr = LinearRegression(featuresCol="features", labelCol="SalePrice")
# creating pipeline in order to use categorical values
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

# then train the model
model = pipeline.fit(train_df)

# here, I want to see coefficients and intercept for the model
lr_model = model.stages[-1] 
print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

# I get the prediction of test dataset
predictions = model.transform(test_df)

# here I display 10 prediction, just to see sample output
predictions.select("features", "SalePrice", "prediction").show(10)

Coefficients: [0.628293170358667,316.397851949599,63.06184265693684,78.47697141768036,3434.056117849626,-6369.570311082219,2113.877046627318,9038.487706701491,6645.39121387521,12931.892373144103,2.38297910463988,17.844168838253985,-24.168280568214154,-38.07350828327549,7578.717583345065,-5755.502545622991,-3120.5041616405492,3136.640069923196,5550.552026931658,-16136.889470203045,-4436.285337764822,11587.15046920337,-4746.005285427499,-10202.432759656742,1784.3081127369464,-14954.786556336836,-2500.72756847851,3669.6649506077406,12579.958961239488,-997.4543596076448,-2902.0818587074477,16099.02704683835,243.13309931786276,-27390.38045075398,-1041.538835009085,1041.5387651108101,591.0703785712692,-103.18923763529551,3756.1112687883733,-38752.48910556882,8872.166132789249,-11249.412642815523,-4746.005285416604,-14764.079313206616,9278.074667326198,-3990.9133705305508,2933.806926754534,16207.223057591063,7135.697972048965,1620.5738169176345,11722.412607052383,-10609.816014588958,-2302.876

In [112]:
evaluator = RegressionEvaluator(
    labelCol="SalePrice", predictionCol="prediction", metricName="rmse"
)

rmse = evaluator.evaluate(predictions)
r2 = RegressionEvaluator(
    labelCol="SalePrice", predictionCol="prediction", metricName="r2"
).evaluate(predictions)

print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R^2: {r2:.3f}")

Root Mean Squared Error (RMSE): 33015.50606953939
R^2: 0.820


here I would like to dimension reduction. I will do PCA
then, I will try what number of PC are the best.

for now, R^2 is 0.82, not perfect but not bad as well

In [113]:
def run_pca_lr(train_df, test_df, k):
    
    pca = PCA(k=k, inputCol="features", outputCol="pcaFeatures")
    lr = LinearRegression(featuresCol="pcaFeatures", labelCol="SalePrice")

    pipeline = Pipeline(stages=indexers + encoders + [assembler, pca, lr])
    model = pipeline.fit(train_df)

    predictions = model.transform(test_df)

    evaluator = RegressionEvaluator(labelCol="SalePrice", predictionCol="prediction", metricName="rmse")
    rmse = evaluator.evaluate(predictions)
    r2 = RegressionEvaluator(labelCol="SalePrice", predictionCol="prediction", metricName="r2").evaluate(predictions)

    # Grab PCA model
    pca_model = [s for s in model.stages if s.__class__.__name__ == "PCAModel"][0]

    print(f"PCA(k={k}) -> RMSE: {rmse:.3f}, R^2: {r2:.3f}")
    return rmse, r2, pca_model.explainedVariance
    
for k in [3, 5, 7, 10,13,16,19]:
    rmse, r2, explained = run_pca_lr(train_df, test_df, k)
    print("Explained variance:", explained)

PCA(k=3) -> RMSE: 39557.231, R^2: 0.742
Explained variance: [0.9910824218059828,0.004867496507828189,0.0032643952543719846]
PCA(k=5) -> RMSE: 38677.315, R^2: 0.753
Explained variance: [0.9910824218059828,0.004867496507828189,0.0032643952543719846,0.00044490499379277917,0.0003203429394499419]
PCA(k=7) -> RMSE: 34252.591, R^2: 0.806
Explained variance: [0.9910824218059828,0.004867496507828189,0.0032643952543719846,0.00044490499379277917,0.0003203429394499419,1.0184212011556446e-05,7.670901530139035e-06]
PCA(k=10) -> RMSE: 32571.554, R^2: 0.825
Explained variance: [0.9910824218059828,0.004867496507828189,0.0032643952543719846,0.00044490499379277917,0.0003203429394499419,1.0184212011556446e-05,7.670901530139035e-06,2.474299728647019e-06,1.1981486142776902e-08,1.1297013713352624e-08]
PCA(k=13) -> RMSE: 30925.110, R^2: 0.842
Explained variance: [0.9910824218059828,0.004867496507828189,0.0032643952543719846,0.00044490499379277917,0.0003203429394499419,1.0184212011556446e-05,7.670901530139035e

it shows K=16, gives 0.846, better result.

Now, checking dimensiton reduction with Lasso, Ridge and  Elastic Net

In [114]:

def run_lr(train_df, test_df, regParam, elasticNetParam, label="SalePrice"):
    
    lr = LinearRegression(
        featuresCol="features",
        labelCol=label,
        regParam=regParam,
        elasticNetParam=elasticNetParam
    )

    # Full pipeline: indexers + encoders + assembler + model
    pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

    # Fit pipeline on train
    model = pipeline.fit(train_df)

    # Transform test with the same pipeline (so categorical encodings exist)
    predictions = model.transform(test_df)

    # Evaluate
    evaluator = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="rmse")
    rmse = evaluator.evaluate(predictions)
    r2 = RegressionEvaluator(labelCol=label, predictionCol="prediction", metricName="r2").evaluate(predictions)

    print(f"ElasticNetParam={elasticNetParam}, regParam={regParam} -> RMSE: {rmse:.3f}, R^2: {r2:.3f}")
    return rmse, r2, model.stages[-1].coefficients

In [115]:
# Ridge (L2)
run_lr(train_df, test_df, regParam=0.1, elasticNetParam=0.0)

ElasticNetParam=0.0, regParam=0.1 -> RMSE: 33006.667, R^2: 0.820


(33006.66681141579,
 0.8202348529170971,
 DenseVector([0.6284, 316.7572, 62.6782, 78.0826, 3443.7198, -6376.6073, 2119.1816, 9042.0648, 6647.1131, 12945.3013, 2.3356, 17.8452, -23.8026, -38.0277, 7592.2114, -5734.2782, -3097.9052, 3041.0343, 5570.9665, -16259.926, -4401.9728, 11601.0956, -4761.9304, -10166.6592, 1796.042, -14954.6294, -2483.1286, 3536.6823, 12556.3008, -991.5317, -2901.8412, 16073.6061, 246.6525, -27380.3778, -1048.0265, 1048.0377, 592.2149, -104.3554, 3756.2491, -38754.4333, 8831.4544, -11179.3841, -4761.9304, -14685.8688, 9222.2278, -3989.9415, 2933.9095, 16187.514, 7129.4937, 1631.519, 11714.644, -10604.1404, -2314.5861, -16696.533, -19260.6844, 475.505, -9566.515, 46335.0834, -12083.9116, -12158.3451, 1359.9255, -388.1488, 18724.0387, -7595.8785, 49844.7476, -2602.2031, 1875.574, -31.1339, 60137.007, 1279.6134, 11627.819, -4721.7053, 2274.7446, 25031.2382, 7895.8279, 429.5319, -363.6384, -583.5422, 9196.2963, -8826.4486, -39020.1777, 1379.3498, 3015.3297, -10691.91

In [116]:
# Lasso (L1)
run_lr(train_df, test_df, regParam=0.1, elasticNetParam=1.0)

ElasticNetParam=1.0, regParam=0.1 -> RMSE: 32430.803, R^2: 0.826


(32430.80276393545,
 0.8264528228623111,
 DenseVector([0.6294, 339.5858, 32.7187, 47.5018, 3690.5009, -6248.903, 1877.6119, 9034.4574, 6778.5321, 13249.0417, 0.9649, 17.6762, 6.559, -40.1462, 8332.6904, -3942.2356, -1874.0874, 4261.1282, 6545.3502, -15146.2309, -2050.6751, 12470.7586, -4228.5282, -9111.9874, 2833.5376, -16369.5233, -1352.7064, 4679.9203, 8717.5623, 774.9237, -2095.9061, 17606.718, 2241.6668, -27730.5064, -1083.7206, 1083.7206, 1407.5963, 549.2214, 4437.9082, -37758.5972, 8877.1048, -11451.8129, -4228.5282, -14757.8427, 9305.0208, -4137.995, 2835.9945, 16249.7964, 6908.9225, 2109.6466, 16103.0058, -10346.5443, -2699.9152, -15080.3931, -18519.9115, 89.9717, -9358.3456, 46035.7606, -12882.1236, -12427.1928, 2512.2975, -775.0373, 19107.5778, -7693.7336, 49862.1185, -1500.2965, 1684.7398, -659.3842, 59829.6395, -1604.5888, 13107.5824, -5418.1274, 3641.9198, 24995.0388, 7431.7259, 1711.6242, -1035.8452, -1208.9406, 8669.2944, -9315.5592, -39230.2639, 732.3485, 2504.7117, -11

In [117]:
# Elastic Net (mix)
run_lr(train_df, test_df, regParam=0.1, elasticNetParam=0.5)

ElasticNetParam=0.5, regParam=0.1 -> RMSE: 32292.124, R^2: 0.828


(32292.124330946077,
 0.827933871195599,
 DenseVector([0.6312, 345.2325, 29.386, 44.5169, 3089.5272, -6277.6532, 1729.8277, 9021.4484, 6751.9636, 13476.1941, -0.3863, 17.8143, 10.728, -35.2858, 8039.0654, -5013.2941, -2520.6206, 4249.5357, 6280.0063, -15355.5203, -2809.8436, 12166.4857, -3118.9071, -6955.3451, 2447.812, -17444.6124, -1464.1456, 5085.7558, 7429.6423, 747.1016, -2279.5843, 17663.806, 2140.7374, -28215.3046, -699.972, 699.972, 783.2559, -81.2432, 3891.2943, -38745.1414, 11100.0535, -10157.4715, -3118.9071, -13616.9764, 8807.7749, -4572.3448, 2371.4898, 15960.2096, 6342.8673, 1391.2109, 14820.1664, -10433.4375, -2617.5263, -14902.2056, -18496.8959, 223.6789, -9468.3158, 46450.4031, -12838.1512, -12175.9576, 2739.2489, -693.52, 19124.0946, -7711.5886, 49731.7823, -1234.1818, 1677.4044, -783.0784, 59959.3046, -1834.8283, 13582.0209, -4846.4758, 4156.2326, 24632.5346, 8229.1552, 1486.0576, -12.4708, -219.7695, 9650.9358, -8188.3018, -38066.6261, 404.8669, 2289.2856, -11876.40

all three method gives same result. R^2 is 0.828. this is less than 0.846 PCA reduction


Now, I decided to not reduce the dimensions. Now, I want to see which models are better perform

In [118]:
pca = PCA(k=16, inputCol="features", outputCol="pcaFeatures")

evaluator_rmse = RegressionEvaluator(labelCol="SalePrice", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="SalePrice", predictionCol="prediction", metricName="r2")

# Linear Regression variants
for regParam in [0.01, 0.1]:
    for elasticNetParam in [0.0, 0.5, 1.0]:  # Ridge, Elastic Net, Lasso
        for withPca in [True, False]:
            lr = LinearRegression(featuresCol="features", labelCol="SalePrice",
                                  regParam=regParam, elasticNetParam=elasticNetParam)
            if withPca:
                pipeline = Pipeline(stages=indexers + encoders + [assembler, pca, lr])
            else:
                pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])
    
            model = pipeline.fit(train_df)
            preds = model.transform(test_df)
            rmse = evaluator_rmse.evaluate(preds)
            r2 = evaluator_r2.evaluate(preds)
            print(f"LinearRegression regParam: {regParam}, elasticNetParam: {elasticNetParam}, withPca: {withPca}, RMSE: {rmse:.3f}, R^2: {r2:.3f}")

# Random Forest
for trees in [20, 50, 70]:
    for withPca in [True, False]:
        rf = RandomForestRegressor(featuresCol="features", labelCol="SalePrice", numTrees=trees)
        if withPca:
            pipeline = Pipeline(stages=indexers + encoders + [assembler, pca, rf])
        else:
            pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])
            
        model = pipeline.fit(train_df)
        preds = model.transform(test_df)
        rmse = evaluator_rmse.evaluate(preds)
        r2 = evaluator_r2.evaluate(preds)
        print(f"RandomForest tree num: {trees}, withPca: {withPca}, RMSE: {rmse:.3f}, R^2: {r2:.3f}")

# Gradient Boosted Trees
for iters in [20, 50]:
    for withPca in [True, False]:
        gbt = GBTRegressor(featuresCol="features", labelCol="SalePrice", maxIter=iters)
        if withPca:
            pipeline = Pipeline(stages=indexers + encoders + [assembler, pca, gbt])
        else:
            pipeline = Pipeline(stages=indexers + encoders + [assembler, gbt])
            
        model = pipeline.fit(train_df)
        preds = model.transform(test_df)
        rmse = evaluator_rmse.evaluate(preds)
        r2 = evaluator_r2.evaluate(preds)
        print(f"GBT iteration: {iters}, withPca: {withPca}, RMSE: {rmse:.3f}, R^2: {r2:.3f}")



LinearRegression regParam: 0.01, elasticNetParam: 0.0, withPca: True, RMSE: 33007.115, R^2: 0.820
LinearRegression regParam: 0.01, elasticNetParam: 0.0, withPca: False, RMSE: 33007.115, R^2: 0.820
LinearRegression regParam: 0.01, elasticNetParam: 0.5, withPca: True, RMSE: 32421.555, R^2: 0.827
LinearRegression regParam: 0.01, elasticNetParam: 0.5, withPca: False, RMSE: 32421.555, R^2: 0.827
LinearRegression regParam: 0.01, elasticNetParam: 1.0, withPca: True, RMSE: 32446.214, R^2: 0.826
LinearRegression regParam: 0.01, elasticNetParam: 1.0, withPca: False, RMSE: 32446.214, R^2: 0.826
LinearRegression regParam: 0.1, elasticNetParam: 0.0, withPca: True, RMSE: 33006.667, R^2: 0.820
LinearRegression regParam: 0.1, elasticNetParam: 0.0, withPca: False, RMSE: 33006.667, R^2: 0.820
LinearRegression regParam: 0.1, elasticNetParam: 0.5, withPca: True, RMSE: 32292.124, R^2: 0.828
LinearRegression regParam: 0.1, elasticNetParam: 0.5, withPca: False, RMSE: 32292.124, R^2: 0.828
LinearRegression re

among, LineerRegression, RandomForest and GBT, best perform model is RandomForest with parameter of 70 tree.
with or without PCA, random forest performs same. so, I choose without PCA

so, I decide to use RandomForest with 70 trees, and make prediction with the competition input

I first load test.csv. this is the file, I need to predict and submit

In [119]:
competition_df = load_data("data/test.csv", False)

competition_df.show(10)

+----+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+----------+--------+------+--------+--------+----------+---------+------------+---------+-------------+--------+-----------+-----+------+----------+----------+----------+------------+-----------+----------+----------+---------+-------+----------+
|  Id|LotArea|YearBuilt|1stFlrSF|2ndFlrSF|FullBath|BedroomAbvGr|TotRmsAbvGrd|OverallQual|OverallCond|GarageCars|GarageArea|TotalBsmtSF|GrLivArea|YearRemodAdd|MSSubClass|MSZoning|Street|LotShape|BldgType|HouseStyle|RoofStyle|Neighborhood|LotConfig|SaleCondition|SaleType|MiscFeature|Fence|PoolQC|PavedDrive|GarageCond|GarageQual|GarageFinish|KitchenQual|Electrical|CentralAir|HeatingQC|Heating|GarageType|
+----+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+----------+--------+------+-----

I first recreate the model with all data (including test and train)

In [120]:
rf = RandomForestRegressor(featuresCol="features", labelCol="SalePrice", numTrees=70)
pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])
model = pipeline.fit(df)


In [121]:
competition_df = load_data("data/test.csv", False)

In [122]:
def set_mean_to_nulls(df: DataFrame):
    for c, dtype in df.dtypes:
        if dtype in ["int", "bigint", "double", "float"]:
            avg_val = df.select(mean(c)).first()[0]
            if avg_val is not None:
                df = df.na.fill({c: int(avg_val)})
    return df

In [123]:
df_clean = competition_df
df_clean = set_mean_to_nulls(df_clean)


In [124]:
df_clean.show(10)

+----+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+----------+--------+------+--------+--------+----------+---------+------------+---------+-------------+--------+-----------+-----+------+----------+----------+----------+------------+-----------+----------+----------+---------+-------+----------+
|  Id|LotArea|YearBuilt|1stFlrSF|2ndFlrSF|FullBath|BedroomAbvGr|TotRmsAbvGrd|OverallQual|OverallCond|GarageCars|GarageArea|TotalBsmtSF|GrLivArea|YearRemodAdd|MSSubClass|MSZoning|Street|LotShape|BldgType|HouseStyle|RoofStyle|Neighborhood|LotConfig|SaleCondition|SaleType|MiscFeature|Fence|PoolQC|PavedDrive|GarageCond|GarageQual|GarageFinish|KitchenQual|Electrical|CentralAir|HeatingQC|Heating|GarageType|
+----+-------+---------+--------+--------+--------+------------+------------+-----------+-----------+----------+----------+-----------+---------+------------+----------+--------+------+-----

In [125]:
preds = model.transform(df_clean)

In [126]:
preds.select("Id","prediction").show(20)

+----+------------------+
|  Id|        prediction|
+----+------------------+
|1461|125855.98607255063|
|1462| 151974.2187932809|
|1463|172355.83974702656|
|1464|182565.44768396617|
|1465| 204709.2525598501|
|1466|181599.61913999717|
|1467| 167725.6573105981|
|1468|171789.19930383176|
|1469| 184754.6619617558|
|1470| 125292.8125323103|
|1471|194126.97035759172|
|1472| 115834.2532629189|
|1473|115839.23170130934|
|1474| 158762.9641863132|
|1475|142498.73050732954|
|1476|385187.47873395035|
|1477| 265911.9333927625|
|1478|330237.52116460557|
|1479| 311371.5384949176|
|1480|427088.47734912846|
+----+------------------+
only showing top 20 rows


In [127]:
# Collect only the two columns you care about
rows = preds.select("Id", "prediction") \
            .rdd.map(lambda r: (r["Id"], r["prediction"])) \
            .collect()

# Convert to Pandas and rename
pdf = pd.DataFrame(rows, columns=["Id", "SalePrice"])

# Write to flat CSV
pdf.to_csv("submission.csv", index=False)

I submit several times. first one gave 0.18132, and last submission gave 0.17644